# Create Pilot Samples

This notebook creates a small, balanced pilot subset from the finalized evaluation dataset. The pilot is intended to test the complete downstream pipeline—audio resolution and loading, inference, transcription, translation, saving, evaluation, error handling, and runtime behaviour—before processing all 4,200 clips.

The default is **3 clips per accent group (21 total)**. Three is appropriate for a fast engineering pilot: it exercises every accent and provides short, medium, and long utterances without implying statistical representativeness. Increase the setting only when broader pipeline stress-testing is needed.

The source dataset is never changed. Selected audio is copied, and the output metadata remains a filtered subset with the exact original columns and column order.

## 1. Imports and configuration

All settings are kept together. Paths are repository-relative and work when the notebook is launched either from `notebooks/` or from the repository root.

In [2]:
%pip install pandas

from pathlib import Path
import hashlib
import shutil

import pandas as pd

PILOT_SAMPLES_PER_GROUP = 3
EXPECTED_ACCENT_GROUPS = 7
EXPECTED_SAMPLES_PER_GROUP = 600

working_dir = Path.cwd().resolve()
if working_dir.name == "notebooks":
    REPO_ROOT = working_dir.parent
elif (working_dir / "data" / "final_sample").is_dir():
    REPO_ROOT = working_dir
else:
    raise RuntimeError(
        "Run this notebook from the repository root or its notebooks/ directory."
    )

FINAL_METADATA_PATH = REPO_ROOT / "data" / "final_sample" / "final_sample_600_per_group.csv"
FINAL_AUDIO_DIR = REPO_ROOT / "data" / "final_sample" / "final_sample_audio"
PILOT_DIR = REPO_ROOT / "data" / "pilot_sample"
PILOT_AUDIO_DIR = PILOT_DIR / "pilot_sample_audio"
PILOT_METADATA_PATH = PILOT_DIR / "pilot_sample_metadata.csv"

print(f"Repository root: {REPO_ROOT}")
print(f"Input metadata: {FINAL_METADATA_PATH.relative_to(REPO_ROOT)}")
print(f"Pilot output: {PILOT_DIR.relative_to(REPO_ROOT)}")

     |████████████████████████████████| 10.8 MB 5.4 MB/s eta 0:00:01
     |████████████████████████████████| 508 kB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 348 kB 73.2 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 70.6 MB/s eta 0:00:01
You should consider upgrading via the '/Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Repository root: /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research
Input metadata: data/final_sample/final_sample_600_per_group.csv
Pilot output: data/pilot_sample


## 2. Load the final metadata

Load the finalized CSV and inspect its actual shape, columns, and a small preview before making any selection assumptions.

In [3]:
if not FINAL_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Final metadata not found: {FINAL_METADATA_PATH}")
if not FINAL_AUDIO_DIR.is_dir():
    raise FileNotFoundError(f"Final audio directory not found: {FINAL_AUDIO_DIR}")

final_df = pd.read_csv(FINAL_METADATA_PATH)
original_columns = final_df.columns.tolist()

print(f"Dataset shape: {final_df.shape}")
print(f"Columns ({len(original_columns)}): {original_columns}")
display(final_df.head(3))

Dataset shape: (4200, 20)
Columns (20): ['path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'primary_accent', 'sentence_norm', 'clip', 'duration[ms]', 'sentence_len_words', 'max_plausible_words', 'sentence_len_chars', 'sample_id']


,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment,primary_accent,sentence_norm,clip,duration[ms],sentence_len_words,max_plausible_words,sentence_len_chars,sample_id
0,common_voice_en_470969.mp3,011490baf5692eb45577c9673689ba64ea5042f8ed216f...,We talked of the side show in the circus.,NaN,2,0,fourties,male_masculine,England English,NaN,en,NaN,England English,we talked of the side show in the circus,common_voice_en_470969.mp3,3624,9,14.496,41,spk_1329
1,common_voice_en_19670219.mp3,1de2845a12b84bcdfad2a954b83e79e5a54064eae0bce7...,"However, after persistent calls, Hall-Jones re...",NaN,2,0,twenties,female_feminine,England English,NaN,en,NaN,England English,however after persistent calls halljones reluc...,common_voice_en_19670219.mp3,9264,12,37.056,107,spk_1385
2,common_voice_en_17262688.mp3,4933d68ff7a8bed9b55641bd748c0a15,What should I say?,NaN,2,0,twenties,male_masculine,England English,NaN,en,NaN,England English,what should i say,common_voice_en_17262688.mp3,2016,4,8.064,18,spk_0337


## 3. Validate the final dataset

Verify the finalized 7 × 600 structure, required selection fields, unique audio paths, and source-audio availability. Failures stop the notebook with a clear message instead of producing a partial pilot.

In [4]:
required_columns = [
    "path", "primary_accent", "duration[ms]",
    "sentence_len_words", "sentence_id", "sample_id",
]
missing_columns = [column for column in required_columns if column not in final_df.columns]
if missing_columns:
    raise ValueError(f"Required metadata columns are missing: {missing_columns}")

if final_df[required_columns].isna().any().any():
    missing_counts = final_df[required_columns].isna().sum()
    raise ValueError(f"Required fields contain missing values:\n{missing_counts[missing_counts > 0]}")
if final_df.duplicated().any():
    raise ValueError(f"Found {int(final_df.duplicated().sum())} duplicated metadata rows.")
if final_df["path"].duplicated().any():
    raise ValueError("Audio paths must be unique before selecting pilot samples.")
if (final_df["duration[ms]"] <= 0).any():
    raise ValueError("All durations must be positive.")

group_counts = final_df["primary_accent"].value_counts().sort_index()
if len(group_counts) != EXPECTED_ACCENT_GROUPS:
    raise ValueError(
        f"Expected {EXPECTED_ACCENT_GROUPS} accent groups, found {len(group_counts)}."
    )
if not group_counts.eq(EXPECTED_SAMPLES_PER_GROUP).all():
    raise ValueError(
        f"Expected {EXPECTED_SAMPLES_PER_GROUP} rows in every accent group:\n{group_counts}"
    )

def safe_relative_audio_path(value):
    relative_path = Path(str(value))
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError(f"Unsafe metadata audio path: {value!r}")
    return relative_path

audio_relative_paths = final_df["path"].map(safe_relative_audio_path)
missing_audio = [
    str(path) for path in audio_relative_paths if not (FINAL_AUDIO_DIR / path).is_file()
]
if missing_audio:
    preview = missing_audio[:10]
    raise FileNotFoundError(
        f"Missing {len(missing_audio)} source audio files. First examples: {preview}"
    )

display(group_counts.rename("samples").to_frame())
print(f"Validated {len(final_df):,} rows and {len(audio_relative_paths):,} source audio files.")

,samples
primary_accent,
England English,600
Filipino,600
Hong Kong English,600
"India and South Asia (India, Pakistan, Sri Lanka)",600
Malaysian English,600
"Southern African (South Africa, Zimbabwe, Namibia)",600
United States English,600


Validated 4,200 rows and 4,200 source audio files.


## 4. Inspect fields useful for selection

The uploaded schema uses `sample_id` values of the form `spk_####`; they repeat across clips and therefore behave as speaker identifiers. Each accent has many distinct IDs. Duration is used for the short/medium/long targets, while sentence length is shown as a useful secondary variation check.

In [5]:
speaker_id_pattern_rate = final_df["sample_id"].astype(str).str.fullmatch(r"spk_\d+").mean()
speaker_like = speaker_id_pattern_rate == 1.0 and final_df["sample_id"].nunique() < len(final_df)
if not speaker_like:
    raise ValueError(
        "sample_id no longer has the inspected repeated spk_#### structure; "
        "review the speaker-diversity logic before continuing."
    )

selection_summary = final_df.groupby("primary_accent").agg(
    rows=("path", "size"),
    distinct_speakers=("sample_id", "nunique"),
    duration_min_ms=("duration[ms]", "min"),
    duration_median_ms=("duration[ms]", "median"),
    duration_max_ms=("duration[ms]", "max"),
    words_min=("sentence_len_words", "min"),
    words_median=("sentence_len_words", "median"),
    words_max=("sentence_len_words", "max"),
)
display(selection_summary)

,rows,distinct_speakers,duration_min_ms,duration_median_ms,duration_max_ms,words_min,words_median,words_max
primary_accent,,,,,,,,
England English,600,398,1728,4284.0,9264,2,9.0,21
Filipino,600,72,1824,4584.0,9336,2,9.0,22
Hong Kong English,600,41,1728,3936.0,9336,2,8.0,20
"India and South Asia (India, Pakistan, Sri Lanka)",600,356,1728,4560.0,9384,2,9.0,26
Malaysian English,600,61,1776,4584.0,9384,2,9.0,22
"Southern African (South Africa, Zimbabwe, Namibia)",600,131,1728,4416.0,9264,2,9.0,22
United States English,600,527,1776,4344.0,9336,2,9.0,23


## 5. Select the pilot samples

For three samples, the deterministic targets are the 20th, 50th, and 80th duration percentiles within each accent. For another configured count, targets are evenly spaced across the same range. The nearest unused clip is chosen with a different speaker whenever possible; `path` provides a stable tie-break. No random sampling is used, so repeated runs select the same rows.

In [6]:
if PILOT_SAMPLES_PER_GROUP < 1:
    raise ValueError("PILOT_SAMPLES_PER_GROUP must be at least 1.")
if PILOT_SAMPLES_PER_GROUP > group_counts.min():
    raise ValueError("Requested more pilot samples than an accent group contains.")

def quantile_positions(sample_count):
    if sample_count == 1:
        return [0.5]
    return [0.2 + (0.6 * index / (sample_count - 1)) for index in range(sample_count)]

def select_accent_samples(group, sample_count):
    targets = group["duration[ms]"].quantile(quantile_positions(sample_count)).tolist()
    selected_indices = []
    used_speakers = set()

    for target in targets:
        candidates = group.loc[~group.index.isin(selected_indices)].copy()
        different_speaker = candidates.loc[~candidates["sample_id"].isin(used_speakers)]
        if not different_speaker.empty:
            candidates = different_speaker

        candidates["_duration_distance"] = (candidates["duration[ms]"] - target).abs()
        chosen_index = candidates.sort_values(
            ["_duration_distance", "path"], kind="mergesort"
        ).index[0]
        selected_indices.append(chosen_index)
        used_speakers.add(group.at[chosen_index, "sample_id"])

    return selected_indices

pilot_indices = []
for _, accent_group in final_df.groupby("primary_accent", sort=True):
    pilot_indices.extend(select_accent_samples(accent_group, PILOT_SAMPLES_PER_GROUP))

pilot_df = final_df.loc[pilot_indices, original_columns].copy()
pilot_df = pilot_df.sort_values(
    ["primary_accent", "duration[ms]", "path"], kind="mergesort"
).reset_index(drop=True)

## 6. Inspect and validate the selection

Check balance, uniqueness, speaker diversity, and membership in the finalized metadata before writing anything. The compact preview makes the duration and sentence-length spread easy to review.

In [7]:
expected_total = EXPECTED_ACCENT_GROUPS * PILOT_SAMPLES_PER_GROUP
pilot_group_counts = pilot_df["primary_accent"].value_counts().sort_index()

assert len(pilot_df) == expected_total, f"Expected {expected_total} rows, found {len(pilot_df)}."
assert pilot_group_counts.eq(PILOT_SAMPLES_PER_GROUP).all(), "Pilot groups are not balanced."
assert not pilot_df["path"].duplicated().any(), "Pilot contains duplicate audio paths."
assert pilot_df[required_columns].notna().all().all(), "Pilot has missing required values."
assert set(pilot_df["path"]).issubset(set(final_df["path"])), "Pilot contains unknown rows."
assert (
    pilot_df.groupby("primary_accent")["sample_id"].nunique()
    == PILOT_SAMPLES_PER_GROUP
).all(), "A group does not have distinct speakers for every pilot clip."
assert pilot_df.columns.tolist() == original_columns, "Metadata columns or order changed."

preview_columns = [
    "primary_accent", "sample_id", "path",
    "duration[ms]", "sentence_len_words", "sentence",
]
display(pilot_df[preview_columns])
display(pilot_group_counts.rename("pilot_samples").to_frame())

,primary_accent,sample_id,path,duration[ms],sentence_len_words,sentence
0,England English,spk_1267,common_voice_en_17298654.mp3,3072,5,Where did they come from?
1,England English,spk_0863,common_voice_en_17266900.mp3,4272,7,"The very next day, he woke refreshed."
2,England English,spk_0736,common_voice_en_19612259.mp3,5952,12,Colman met the duo when they were all students...
3,Filipino,spk_0110,common_voice_en_15713.mp3,3288,6,Insert your card to identify yourself.
4,Filipino,spk_0900,common_voice_en_17281012.mp3,4584,11,We would prefer for your search to be carried ...
5,Filipino,spk_0464,common_voice_en_18674001.mp3,6216,8,This enables band offset engineering in nanosc...
6,Hong Kong English,spk_1043,common_voice_en_17787313.mp3,2856,8,"A tongue of honey, a heart of gall."
7,Hong Kong English,spk_0294,common_voice_en_100326.mp3,3936,4,The kangaroo hopped noisily.
8,Hong Kong English,spk_0028,common_voice_en_18978486.mp3,5712,8,"Others prohibit all animals that creep, includ..."
9,"India and South Asia (India, Pakistan, Sri Lanka)",spk_0341,common_voice_en_17311041.mp3,3216,9,Did anyone else share this lab at the time?


,pilot_samples
primary_accent,
England English,3
Filipino,3
Hong Kong English,3
"India and South Asia (India, Pakistan, Sri Lanka)",3
Malaysian English,3
"Southern African (South Africa, Zimbabwe, Namibia)",3
United States English,3


## 7. Create the output directories

Create the pilot directories if needed. Re-running this step is safe. The guards ensure the pilot audio directory cannot be confused with the finalized source directory.

In [8]:
if PILOT_AUDIO_DIR.resolve() == FINAL_AUDIO_DIR.resolve():
    raise RuntimeError("Pilot and final audio directories must be different.")
if PILOT_AUDIO_DIR.parent.resolve() != PILOT_DIR.resolve():
    raise RuntimeError("Pilot audio directory must be directly inside PILOT_DIR.")

PILOT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {PILOT_AUDIO_DIR}")

Output directory ready: /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/data/pilot_sample/pilot_sample_audio


## 8. Copy selected audio

First verify every selected source. Then synchronize the pilot audio directory: stale files from an earlier configuration are removed **only from the pilot output**, and selected files are copied with `copy2`. Existing selected files are safely overwritten, making re-runs idempotent while preserving filenames and any relative subdirectories.

In [9]:
selected_relative_paths = [safe_relative_audio_path(value) for value in pilot_df["path"]]
selected_sources = [FINAL_AUDIO_DIR / path for path in selected_relative_paths]
missing_selected_sources = [str(path) for path in selected_sources if not path.is_file()]
if missing_selected_sources:
    raise FileNotFoundError(
        f"Selected source audio is missing: {missing_selected_sources[:10]}"
    )

def file_digest(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

metadata_digest_before = file_digest(FINAL_METADATA_PATH)
source_signatures_before = {
    path: (path.stat().st_size, path.stat().st_mtime_ns) for path in selected_sources
}

expected_relative_paths = set(selected_relative_paths)
existing_output_files = [path for path in PILOT_AUDIO_DIR.rglob("*") if path.is_file()]
stale_output_files = [
    path for path in existing_output_files
    if path.relative_to(PILOT_AUDIO_DIR) not in expected_relative_paths
]
for stale_path in stale_output_files:
    stale_path.unlink()

for source_path, relative_path in zip(selected_sources, selected_relative_paths):
    destination_path = PILOT_AUDIO_DIR / relative_path
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination_path)

for directory in sorted(
    (path for path in PILOT_AUDIO_DIR.rglob("*") if path.is_dir()),
    key=lambda path: len(path.parts),
    reverse=True,
):
    if not any(directory.iterdir()):
        directory.rmdir()

print(f"Copied {len(selected_sources)} audio files; removed {len(stale_output_files)} stale pilot files.")

Copied 21 audio files; removed 0 stale pilot files.


## 9. Save pilot metadata

Write exactly the selected rows with `index=False`. No columns are added or renamed, their order is unchanged, and the relative `path` values continue to resolve inside `pilot_sample_audio/`.

In [10]:
pilot_df.to_csv(PILOT_METADATA_PATH, index=False)
print(f"Saved {len(pilot_df)} rows to {PILOT_METADATA_PATH}")

Saved 21 rows to /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/data/pilot_sample/pilot_sample_metadata.csv


## 10. Final validation

Reload the saved CSV and verify its schema, balance, exact audio-file set, source/destination content, valid paths, and unchanged source metadata and selected source files.

In [11]:
if not PILOT_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Pilot metadata was not created: {PILOT_METADATA_PATH}")

saved_pilot_df = pd.read_csv(PILOT_METADATA_PATH)
output_files = [path for path in PILOT_AUDIO_DIR.rglob("*") if path.is_file()]
output_relative_paths = {path.relative_to(PILOT_AUDIO_DIR) for path in output_files}
saved_relative_paths = {
    safe_relative_audio_path(value) for value in saved_pilot_df["path"]
}
source_signatures_after = {
    path: (path.stat().st_size, path.stat().st_mtime_ns) for path in selected_sources
}

assert saved_pilot_df.columns.tolist() == original_columns, "Saved metadata schema changed."
assert len(saved_pilot_df) == expected_total, "Saved metadata row count is incorrect."
assert (
    saved_pilot_df["primary_accent"].value_counts().sort_index()
    == pilot_group_counts
).all(), "Saved accent distribution is incorrect."
assert saved_relative_paths == expected_relative_paths, "Saved metadata paths changed."
assert output_relative_paths == expected_relative_paths, "Pilot audio directory has missing or extra files."
assert all((PILOT_AUDIO_DIR / path).is_file() for path in saved_relative_paths), "A pilot path does not resolve."
assert file_digest(FINAL_METADATA_PATH) == metadata_digest_before, "Final metadata was modified."
assert source_signatures_after == source_signatures_before, "A selected source audio file was modified."
assert all(
    file_digest(FINAL_AUDIO_DIR / path) == file_digest(PILOT_AUDIO_DIR / path)
    for path in expected_relative_paths
), "At least one copied audio file differs from its source."

print("Pilot dataset created successfully\n")
print(f"Accent groups:       {len(pilot_group_counts)}")
print(f"Samples per group:   {PILOT_SAMPLES_PER_GROUP}")
print(f"Metadata rows:       {len(saved_pilot_df)}")
print(f"Audio files copied:  {len(output_files)}")
print("Missing files:       0")

Pilot dataset created successfully

Accent groups:       7
Samples per group:   3
Metadata rows:       21
Audio files copied:  21
Missing files:       0


## 11. Preview the final pilot dataset

Display the saved subset in a compact form for a final human check. This pilot is deliberately small and varied for pipeline debugging; it must not be reported as a statistically representative evaluation subset.

In [12]:
display(saved_pilot_df[preview_columns])

,primary_accent,sample_id,path,duration[ms],sentence_len_words,sentence
0,England English,spk_1267,common_voice_en_17298654.mp3,3072,5,Where did they come from?
1,England English,spk_0863,common_voice_en_17266900.mp3,4272,7,"The very next day, he woke refreshed."
2,England English,spk_0736,common_voice_en_19612259.mp3,5952,12,Colman met the duo when they were all students...
3,Filipino,spk_0110,common_voice_en_15713.mp3,3288,6,Insert your card to identify yourself.
4,Filipino,spk_0900,common_voice_en_17281012.mp3,4584,11,We would prefer for your search to be carried ...
5,Filipino,spk_0464,common_voice_en_18674001.mp3,6216,8,This enables band offset engineering in nanosc...
6,Hong Kong English,spk_1043,common_voice_en_17787313.mp3,2856,8,"A tongue of honey, a heart of gall."
7,Hong Kong English,spk_0294,common_voice_en_100326.mp3,3936,4,The kangaroo hopped noisily.
8,Hong Kong English,spk_0028,common_voice_en_18978486.mp3,5712,8,"Others prohibit all animals that creep, includ..."
9,"India and South Asia (India, Pakistan, Sri Lanka)",spk_0341,common_voice_en_17311041.mp3,3216,9,Did anyone else share this lab at the time?
